# Phase 2 - Marketing Funnel and Traffic Source ROI
Notebook nay phan tich funnel conversion, cancel behavior, va ROI theo traffic source tu agg_funnel_monthly.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED = ROOT / 'data' / 'processed'

agg_funnel = pd.read_parquet(PROCESSED / 'agg_funnel_monthly.parquet')
agg_funnel['year_month'] = pd.to_datetime(agg_funnel['year_month'], errors='coerce')

print(agg_funnel.shape)
agg_funnel.head()

In [ ]:
funnel_summary = (
    agg_funnel.groupby('event_type', as_index=False)
    .agg(total_events=('event_count', 'sum'), total_users=('unique_users', 'sum'))
)
funnel_summary.to_csv(PROCESSED / 'funnel_analysis.csv', index=False)

stage_map = {'product': 'Product View', 'cart': 'Cart', 'purchase': 'Purchase'}
funnel_summary['stage'] = funnel_summary['event_type'].map(stage_map).fillna(funnel_summary['event_type'])

def get_events(name):
    rows = funnel_summary[funnel_summary['event_type'] == name]['total_events']
    return float(rows.iloc[0]) if len(rows) else 0.0

product_total = get_events('product')
cart_total = get_events('cart')
purchase_total = get_events('purchase')
cancel_total = get_events('cancel')

product_to_cart = (cart_total / product_total * 100) if product_total else np.nan
cart_to_purchase = (purchase_total / cart_total * 100) if cart_total else np.nan
product_to_purchase = (purchase_total / product_total * 100) if product_total else np.nan
abandonment_proxy = (cancel_total / (cart_total + purchase_total + cancel_total) * 100) if (cart_total + purchase_total + cancel_total) else np.nan

print(f'Product -> Cart: {product_to_cart:.2f}%')
print(f'Cart -> Purchase: {cart_to_purchase:.2f}%')
print(f'Product -> Purchase: {product_to_purchase:.2f}%')
print(f'Cancel proxy rate: {abandonment_proxy:.2f}%')

stages = ['Product View', 'Cart', 'Purchase']
values = [product_total, cart_total, purchase_total]
fig = go.Figure(go.Funnel(y=stages, x=values, textinfo='value+percent previous'))
fig.update_layout(title='Sales Funnel (2019-2023)')
fig.show()

funnel_summary.sort_values('total_events', ascending=False)

In [ ]:
funnel_by_source = (
    agg_funnel.groupby(['traffic_source', 'event_type'], as_index=False)
    .agg(event_count=('event_count', 'sum'), unique_users=('unique_users', 'sum'))
)

purchase_source = funnel_by_source[funnel_by_source['event_type'] == 'purchase'].copy()
purchase_source['purchase_events_per_user'] = (
    purchase_source['event_count'] / purchase_source['unique_users'].replace(0, np.nan)
)
purchase_source = purchase_source.sort_values('purchase_events_per_user', ascending=False)
purchase_source.to_csv(PROCESSED / 'traffic_source_performance.csv', index=False)

fig = px.bar(
    purchase_source,
    x='traffic_source',
    y='purchase_events_per_user',
    title='Purchase Events per User by Traffic Source'
)
fig.show()

purchase_source

In [ ]:
monthly_trend = (
    agg_funnel[agg_funnel['event_type'].isin(['product', 'cart', 'purchase'])]
    .groupby(['year_month', 'event_type'], as_index=False)['event_count']
    .sum()
)

fig = px.line(
    monthly_trend,
    x='year_month',
    y='event_count',
    color='event_type',
    title='Monthly Funnel Event Trend'
)
fig.show()

## Key Takeaways
- Funnel conversion metrics are exported to data/processed/funnel_analysis.csv.
- Source-level purchase intensity is exported to data/processed/traffic_source_performance.csv.
- Charts can be reused directly in reporting or Power BI narrative validation.